# RunwayMC — PCAP Project Demo
## Robust Aircraft Runway Scheduling Under Uncertainty
### Parallel Implementations: Serial vs OpenMP (CPU) vs CUDA (GPU)

---

**Parallel Computing Concepts Demonstrated:**
1. **The Problem:** Aircraft Landing & Takeoff Scheduling is NP-hard with stochastic delays and wake-turbulence separation constraints.
2. **The Algorithm:** Simulated Annealing (SA) with $C$ Markov chains evaluated across $S$ Monte Carlo noise scenarios.
3. **Three Parallelization Strategies:**
   - **Serial:** 1 thread, $C$ sequential chains.
   - **OpenMP:** `#pragma omp parallel for` distributes chains across all CPU cores.
   - **CUDA:** Each GPU thread block runs one SA chain; warp-shuffle reductions aggregate scenario costs.

> **Disclaimer:** Academic simulation only. Not for real air-traffic use.

## Step 1: Verify GPU & CUDA Toolkit
> **IMPORTANT:** Before running, go to **Runtime → Change runtime type → T4 GPU** in Colab.

In [ ]:
!nvidia-smi
!nvcc --version
!nproc

## Step 2: Clone Repository

In [ ]:
import os
import subprocess

REPO = '/content/PCAP'

if not os.path.exists(REPO):
    subprocess.run(['git', 'clone', 'https://github.com/manoharadimalla/PCAP.git', REPO], check=True)
else:
    subprocess.run(['git', '-C', REPO, 'fetch', 'origin'], check=True)
    subprocess.run(['git', '-C', REPO, 'reset', '--hard', 'origin/main'], check=True)

os.chdir(REPO)
print('Working directory:', os.getcwd())
!git log --oneline -3

## Step 3: Build with CUDA + OpenMP

In [ ]:
import subprocess, sys

# Always clean build to avoid stale cache issues
subprocess.run(['rm', '-rf', 'build'], check=True)

r = subprocess.run(
    ['cmake', '-B', 'build', '-DCMAKE_BUILD_TYPE=Release', '-DUSE_CUDA=ON', '-DUSE_OPENMP=ON'],
    capture_output=True, text=True
)
print(r.stdout[-3000:] if len(r.stdout) > 3000 else r.stdout)
if r.returncode != 0:
    print('CMAKE CONFIGURE FAILED:\n', r.stderr[-2000:])
    sys.exit(1)

import multiprocessing
r2 = subprocess.run(
    ['cmake', '--build', 'build', '--config', 'Release', f'-j{multiprocessing.cpu_count()}'],
    capture_output=True, text=True
)
# Show only last part of build output
out = r2.stdout + r2.stderr
print(out[-4000:] if len(out) > 4000 else out)

if r2.returncode != 0:
    print('BUILD FAILED!')
    sys.exit(1)
else:
    print('\n✅ BUILD SUCCEEDED!')

# Confirm binary exists and report GPU availability
assert os.path.exists('build/runway'), 'Binary not found after build!'
info = subprocess.run(['./build/runway', 'info'], capture_output=True, text=True)
print('\nRunway binary info:', info.stdout)

## Step 4: Run Catch2 Test Suite

In [ ]:
!ctest --test-dir build --output-on-failure

## Step 5: Interactive Demo — Solve a Flight Batch with All 3 Implementations

**You can change the input parameters in the cell below!**
- `INSTANCE` — which CSV file to use (see `data/instances/`)
- `CHAINS` — number of parallel SA chains
- `SCENARIOS` — number of Monte Carlo delay/weather scenarios
- `ITERS` — number of SA iterations per run
- `OMP_THREADS` — number of CPU threads for OpenMP

In [ ]:
# =============================================
# CONFIGURE YOUR DEMO INPUT HERE
# =============================================
INSTANCE    = 'data/instances/demo_interactive.csv'  # change to n30_r12_s42.csv for bigger run
CHAINS      = 16      # number of parallel Markov chains
SCENARIOS   = 1024    # Monte Carlo scenarios per evaluation
ITERS       = 1000    # SA iterations
OMP_THREADS = 2       # CPU threads for OpenMP
SEED        = 42
# =============================================

import json, subprocess, os
import pandas as pd

os.makedirs('results', exist_ok=True)

# Show input flights
print('=== INPUT FLIGHT BATCH ===')
df = pd.read_csv(INSTANCE)
display(df)
print(f'Total flights: {len(df)}')

EXE = './build/runway'

def solve(impl, out_file, extra_args=[]):
    cmd = [
        EXE, 'solve',
        '--instance', INSTANCE,
        '--impl', impl,
        '--chains', str(CHAINS),
        '--scenarios', str(SCENARIOS),
        '--iters', str(ITERS),
        '--seed', str(SEED),
        '--out', out_file,
        '--progress'
    ] + extra_args
    print(f'\n--- Running {impl.upper()} solve ---')
    r = subprocess.run(cmd, capture_output=True, text=True)
    print(r.stdout)
    if r.returncode != 0:
        print(f'ERROR: {r.stderr}')
        return None
    with open(out_file) as f:
        return json.load(f)

res_ser  = solve('serial', 'results/demo_serial.json')
res_omp  = solve('omp',    'results/demo_omp.json',
                 ['--omp-mode', 'chains', '--threads', str(OMP_THREADS)])
res_cuda = solve('cuda',   'results/demo_cuda.json')

In [ ]:
# ============================================================
# Display Performance Summary Table
# ============================================================
import pandas as pd

t_ser  = res_ser['timings']['kernel_ms']
t_omp  = res_omp['timings']['kernel_ms']
t_cuda = res_cuda['timings']['kernel_ms']

fcfs_cost = res_ser['metrics']['fcfs']['mean_cost']
best_cost = res_ser['metrics']['best']['mean_cost']
savings   = (fcfs_cost - best_cost) / fcfs_cost * 100

rows = [
    {'Implementation':  'Serial (1 thread)',
     'Kernel Time (ms)': f'{t_ser:.1f}',
     'Speedup vs Serial': '1.00x',
     'Sims / Second': f"{res_ser['timings']['sim_per_s']:>12,.0f}"},
    {'Implementation': f'OpenMP ({OMP_THREADS} threads)',
     'Kernel Time (ms)': f'{t_omp:.1f}',
     'Speedup vs Serial': f'{t_ser/t_omp:.2f}x',
     'Sims / Second': f"{res_omp['timings']['sim_per_s']:>12,.0f}"},
    {'Implementation': 'CUDA GPU (Tesla T4)',
     'Kernel Time (ms)': f'{t_cuda:.1f}',
     'Speedup vs Serial': f'{t_ser/t_cuda:.2f}x',
     'Sims / Second': f"{res_cuda['timings']['sim_per_s']:>12,.0f}"},
]

print('\n' + '='*65)
print('         PARALLEL PERFORMANCE COMPARISON — PCAP PROJECT')
print('='*65)
display(pd.DataFrame(rows))

print('\nSchedule Quality (evaluated on 20,000 unseen test scenarios):')
print(f'  Baseline FCFS delay cost  : {fcfs_cost:.1f} seconds')
print(f'  RunwayMC optimized cost   : {best_cost:.1f} seconds')
print(f'  Delay reduction achieved  : {savings:.1f}%  ← This is the AI benefit!')

## Step 6: Full Benchmark Suite (E1–E5) with CUDA Auto-Detection

In [ ]:
!python3 bench/run_bench.py --all
!python3 scripts/make_plots.py

## Step 7: Display All Benchmark Figures

In [ ]:
from IPython.display import Image, display
import glob

figures = sorted(glob.glob('results/figures/*.png'))
if not figures:
    print('No figures found — run Step 6 first.')
else:
    for fig_path in figures:
        print(f'\n=== {os.path.basename(fig_path)} ===')
        display(Image(filename=fig_path, width=720))